# Feature Type Analysis of loan_data.csv

This notebook inspects `DPG/datasets/credit_card_approval/loan_data.csv` and infers the **semantic type** of every column: numerical, categorical (low-cardinality), high-cardinality categorical, binary, datetime, id, etc.

The classification uses both pandas dtypes and cardinality heuristics.


In [23]:
import pandas as pd
import numpy as np

CSV_PATH = r"/root/gitgud/temp/DPG/datasets/credit_card_approval/loan_data.csv"

df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


Shape: (1061, 14)
Columns: ['person_age', 'person_gender', 'person_education', 'person_income', 'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'previous_loan_defaults_on_file', 'loan_status']


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [24]:
print("dtypes:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isna().sum())

print("\nCardinality per column:")
print(df.nunique().sort_values())


dtypes:
person_age                        float64
person_gender                         str
person_education                      str
person_income                     float64
person_emp_exp                      int64
person_home_ownership                 str
loan_amnt                         float64
loan_intent                           str
loan_int_rate                     float64
loan_percent_income               float64
cb_person_cred_hist_length        float64
credit_score                        int64
previous_loan_defaults_on_file        str
loan_status                         int64
dtype: object

Missing values per column:
person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb

In [25]:
def classify_feature(series: pd.Series) -> str:
    """Return a semantic type for a pandas Series."""
    n = len(series)
    nunique = series.nunique(dropna=True)
    dtype = series.dtype
    is_numeric = pd.api.types.is_numeric_dtype(dtype)
    is_bool = pd.api.types.is_bool_dtype(dtype)
    is_datetime = pd.api.types.is_datetime64_any_dtype(dtype)

    if is_datetime:
        return "datetime"
    if is_bool or (is_numeric and nunique == 2):
        return "binary"
    if nunique <= 1:
        return "constant"

    # cardinality-relative heuristics
    rel = nunique / max(n, 1)

    if is_numeric:
        if rel > 0.5 and nunique > 50:
            return "numerical_continuous"  # likely id-like or continuous
        return "numerical_discrete"

    # non-numeric
    if nunique == 2:
        return "binary"
    if nunique <= 20 or rel < 0.05:
        return "categorical"
    if rel > 0.5:
        return "high_cardinality_categorical"
    return "categorical"


summary = []
for col in df.columns:
    s = df[col]
    feat_type = classify_feature(s)
    summary.append({
        "feature": col,
        "pandas_dtype": str(s.dtype),
        "n_unique": s.nunique(dropna=True),
        "n_missing": int(s.isna().sum()),
        "example_values": list(s.dropna().unique()[:3]),
        "feature_type": feat_type,
    })

feature_summary = pd.DataFrame(summary)
feature_summary


,feature,pandas_dtype,n_unique,n_missing,example_values,feature_type
0,person_age,float64,8,0,"[22.0, 21.0, 25.0]",numerical_discrete
1,person_gender,str,2,0,"[female, male]",binary
2,person_education,str,5,0,"[Master, High School, Bachelor]",categorical
3,person_income,float64,1051,0,"[71948.0, 12282.0, 12438.0]",numerical_continuous
4,person_emp_exp,int64,13,0,"[0, 3, 1]",numerical_discrete
5,person_home_ownership,str,4,0,"[RENT, OWN, MORTGAGE]",categorical
6,loan_amnt,float64,161,0,"[35000.0, 1000.0, 5500.0]",numerical_discrete
7,loan_intent,str,6,0,"[PERSONAL, EDUCATION, MEDICAL]",categorical
8,loan_int_rate,float64,208,0,"[16.02, 11.14, 12.87]",numerical_discrete
9,loan_percent_income,float64,60,0,"[0.49, 0.08, 0.44]",numerical_discrete


In [26]:
type_counts = feature_summary["feature_type"].value_counts()
print("Feature-type distribution:")
print(type_counts)


Feature-type distribution:
feature_type
numerical_discrete      7
binary                  3
categorical             3
numerical_continuous    1
Name: count, dtype: int64


In [27]:
numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 10]
desc = df[numeric_cols].describe().T
desc[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]


,count,mean,std,min,25%,50%,75%,max
person_income,1061.0,93125.444863,72492.603802,12282.00,33227.00,75802.00,111729.00,600891.00
person_emp_exp,1061.0,2.062205,7.022061,0.00,0.00,1.00,3.00,125.00
loan_amnt,1061.0,17121.913289,8683.483659,500.00,8500.00,20000.00,24000.00,35000.00
loan_int_rate,1061.0,12.061310,3.032398,5.42,10.38,11.58,14.22,20.00
loan_percent_income,1061.0,0.224958,0.111812,0.00,0.14,0.22,0.30,0.66
credit_score,1061.0,625.639962,49.450551,447.00,593.00,634.00,662.00,807.00


In [28]:
import os

# Build a small dummy dataset: 2 categorical + 2 numerical + target class
KEEP_FEATURES = [
    "person_gender",        # categorical (binary)
    "person_home_ownership",# categorical
    # "person_income",        # numerical
    #"loan_amnt",           # numerical
    "person_age",           # numerical
    "loan_status",          # target / class
]

RANDOM_STATE = 42
N_SAMPLES = 30

dummy_df = (
    df[KEEP_FEATURES]
    .sample(n=N_SAMPLES, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

# Persist to disk
out_dir = r"d:\Lucas\UNITS_MAGISTRALE\THESIS\DPG\datasets\dummy_dataset"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "dummy_dataset.csv")
dummy_df.to_csv(out_path, index=False)

print(f"Saved {len(dummy_df)} rows to: {out_path}")
dummy_df


Saved 30 rows to: d:\Lucas\UNITS_MAGISTRALE\THESIS\DPG\datasets\dummy_dataset/dummy_dataset.csv


,person_gender,person_home_ownership,person_age,loan_status
0,male,RENT,25.0,0
1,female,RENT,25.0,1
2,male,RENT,22.0,1
3,female,RENT,26.0,0
4,male,RENT,26.0,0
5,male,RENT,26.0,0
6,female,MORTGAGE,22.0,1
7,female,RENT,24.0,1
8,male,RENT,23.0,0
9,male,RENT,23.0,1
